In [1]:
import sys
from pathlib import Path
from time import perf_counter

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
DATA_DIR = ROOT / "datasets" / "final" / "ml"
MODEL_DIR = ROOT / "models" / "baseline"
RESULTS_DIR = ROOT / "results" / "model_training" / "baseline"

RANDOM_STATE = 1
N_SPLITS = 5
N_JOBS = -1

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))
from src.utils.features import split_features

reference_df = pd.read_parquet(DATA_DIR / "cicids2017_train.parquet")
numerical_features, categorical_features = split_features(reference_df)
feature_columns = numerical_features + categorical_features
del reference_df

#avalia cada modelo com os hiperparâmetros padrão por validação cruzada estratificada de 5 folds,
# registra métricas e tempos e, depois, treina o pipeline final com todos os dados de treinamento

TRAINING_CONFIGS = {
    "cicids2017": ["cicids2017"],
    "unsw_nb15": ["unsw_nb15"],
    "iot23": ["iot23"],
    "cicids2017_unsw_nb15": ["cicids2017", "unsw_nb15"],
    "cicids2017_iot23": ["cicids2017", "iot23"],
    "unsw_nb15_iot23": ["unsw_nb15", "iot23"],
}

MODELS_TO_TRAIN = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]

In [2]:
def build_classifier(model_name):
    # n_jobs=1 evita paralelismo aninhado; o paralelismo ocorre entre os folds.
    classifiers = {
        "decision_tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "random_forest": RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "xgboost": XGBClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1,
            eval_metric="logloss",
            tree_method="hist",
        ),
        "linear_svm": LinearSVC(random_state=RANDOM_STATE),
        "mlp": MLPClassifier(random_state=RANDOM_STATE),
    }
    return classifiers[model_name]


def build_pipeline(classifier):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", StandardScaler(), numerical_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ]
    )


cross_validation = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = {
    "precision_attack": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_attack": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1_attack": make_scorer(f1_score, pos_label=1, zero_division=0),
    "precision_benign": make_scorer(precision_score, pos_label=0, zero_division=0),
    "recall_benign": make_scorer(recall_score, pos_label=0, zero_division=0),
    "f1_benign": make_scorer(f1_score, pos_label=0, zero_division=0),
    "f1_macro": make_scorer(
        f1_score,
        average="macro",
        labels=[0, 1],
        zero_division=0,
    ),
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
}

In [3]:
def load_training_data(dataset_names):
    columns_to_load = feature_columns + ["label_binary"]
    dataframes = [
        pd.read_parquet(
            DATA_DIR / f"{dataset_name}_train.parquet",
            columns=columns_to_load,
        )
        for dataset_name in dataset_names
    ]
    training_df = pd.concat(dataframes, ignore_index=True)
    features = training_df[feature_columns]
    target = training_df["label_binary"].eq("ATTACK").astype("int8")
    return features, target


def train_baseline(model_name, training_name, dataset_names):
    features, target = load_training_data(dataset_names)
    model_results_dir = RESULTS_DIR / model_name
    model_output_dir = MODEL_DIR / model_name
    model_results_dir.mkdir(parents=True, exist_ok=True)
    model_output_dir.mkdir(parents=True, exist_ok=True)

    cv_pipeline = build_pipeline(build_classifier(model_name))
    cv_start = perf_counter()
    cv_results = cross_validate(
        cv_pipeline,
        features,
        target,
        scoring=scoring,
        cv=cross_validation,
        n_jobs=N_JOBS,
        return_train_score=False,
        error_score="raise",
    )
    cv_elapsed_seconds = perf_counter() - cv_start

    fold_results = pd.DataFrame(
        {
            "model": model_name,
            "training_method": "baseline",
            "training_source": training_name,
            "training_rows": len(features),
            "n_splits": N_SPLITS,
            "random_state": RANDOM_STATE,
            "fold": range(1, N_SPLITS + 1),
            "fit_time_seconds": cv_results["fit_time"],
            "score_time_seconds": cv_results["score_time"],
        }
    )
    for metric in scoring:
        fold_results[metric] = cv_results[f"test_{metric}"]

    fold_results.to_csv(
        model_results_dir / f"{training_name}_folds.csv",
        index=False,
    )

    final_pipeline = build_pipeline(build_classifier(model_name))
    final_fit_start = perf_counter()
    final_pipeline.fit(features, target)
    final_fit_time_seconds = perf_counter() - final_fit_start

    model_path = model_output_dir / f"{training_name}.joblib"
    joblib.dump(final_pipeline, model_path, compress=3)

    total_elapsed_seconds = cv_elapsed_seconds + final_fit_time_seconds
    result = {
        "model": model_name,
        "training_method": "baseline",
        "training_source": training_name,
        "training_rows": len(features),
        "n_splits": N_SPLITS,
        "random_state": RANDOM_STATE,
        "params": "{}",
        "elapsed_seconds": total_elapsed_seconds,
        "elapsed_minutes": total_elapsed_seconds / 60,
        "cv_elapsed_seconds": cv_elapsed_seconds,
        "refit_time_seconds": final_fit_time_seconds,
        "fit_time_mean": fold_results["fit_time_seconds"].mean(),
        "fit_time_std": fold_results["fit_time_seconds"].std(ddof=0),
        "score_time_mean": fold_results["score_time_seconds"].mean(),
        "score_time_std": fold_results["score_time_seconds"].std(ddof=0),
        "model_size_mb": model_path.stat().st_size / 1024 ** 2,
        "model_file": model_path.name,
    }
    for metric in scoring:
        result[f"{metric}_mean"] = fold_results[metric].mean()
        result[f"{metric}_std"] = fold_results[metric].std(ddof=0)

    return result

In [4]:
all_results = []

for model_name in MODELS_TO_TRAIN:
    model_results = []

    for training_name, dataset_names in TRAINING_CONFIGS.items():
        print(f"Modelo: {model_name} | origem do treinamento: {training_name}")
        result = train_baseline(
            model_name,
            training_name,
            dataset_names,
        )
        model_results.append(result)
        all_results.append(result)

    pd.DataFrame(model_results).to_csv(
        RESULTS_DIR / model_name / "summary.csv",
        index=False,
    )

all_results_df = pd.DataFrame(all_results)
all_results_df.to_csv(RESULTS_DIR / "summary.csv", index=False)
display(all_results_df)

Modelo: decision_tree | origem do treinamento: cicids2017
Modelo: decision_tree | origem do treinamento: unsw_nb15
Modelo: decision_tree | origem do treinamento: iot23
Modelo: decision_tree | origem do treinamento: cicids2017_unsw_nb15
Modelo: decision_tree | origem do treinamento: cicids2017_iot23
Modelo: decision_tree | origem do treinamento: unsw_nb15_iot23
Modelo: random_forest | origem do treinamento: cicids2017
Modelo: random_forest | origem do treinamento: unsw_nb15
Modelo: random_forest | origem do treinamento: iot23
Modelo: random_forest | origem do treinamento: cicids2017_unsw_nb15
Modelo: random_forest | origem do treinamento: cicids2017_iot23
Modelo: random_forest | origem do treinamento: unsw_nb15_iot23
Modelo: xgboost | origem do treinamento: cicids2017
Modelo: xgboost | origem do treinamento: unsw_nb15
Modelo: xgboost | origem do treinamento: iot23
Modelo: xgboost | origem do treinamento: cicids2017_unsw_nb15
Modelo: xgboost | origem do treinamento: cicids2017_iot23
Mode

C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Modelo: mlp | origem do treinamento: unsw_nb15_iot23


,model,training_method,training_source,training_rows,n_splits,random_state,params,elapsed_seconds,elapsed_minutes,cv_elapsed_seconds,...,recall_benign_mean,recall_benign_std,f1_benign_mean,f1_benign_std,f1_macro_mean,f1_macro_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std
0,decision_tree,baseline,cicids2017,235490,5,1,{},6.396218,0.106604,4.713621,...,0.995377,0.000327,0.996133,0.000145,0.990361,0.000363,0.993817,0.000232,0.991478,0.000652
1,decision_tree,baseline,unsw_nb15,235490,5,1,{},6.355217,0.105920,4.412126,...,0.987659,0.000303,0.988346,0.000334,0.970945,0.000858,0.981367,0.000538,0.971928,0.001267
2,decision_tree,baseline,iot23,235490,5,1,{},3.376987,0.056283,2.807623,...,0.999719,0.000055,0.999066,0.000124,0.997660,0.000312,0.998505,0.000199,0.996685,0.000457
3,decision_tree,baseline,cicids2017_unsw_nb15,470980,5,1,{},9.821428,0.163690,5.311171,...,0.991305,0.000339,0.992004,0.000297,0.980067,0.000740,0.987216,0.000474,0.981082,0.000773
4,decision_tree,baseline,cicids2017_iot23,470980,5,1,{},6.810080,0.113501,3.853531,...,0.997317,0.000126,0.997514,0.000076,0.993790,0.000189,0.996023,0.000121,0.994083,0.000230
5,decision_tree,baseline,unsw_nb15_iot23,470980,5,1,{},5.685558,0.094759,3.185804,...,0.993689,0.000175,0.993723,0.000222,0.984309,0.000562,0.989957,0.000356,0.984360,0.000793
6,random_forest,baseline,cicids2017,235490,5,1,{},49.531468,0.825524,27.368668,...,0.995409,0.000373,0.996808,0.000188,0.992065,0.000466,0.994900,0.000300,0.994137,0.000566
7,random_forest,baseline,unsw_nb15,235490,5,1,{},50.014362,0.833573,27.625963,...,0.986395,0.000383,0.991455,0.000273,0.979066,0.000663,0.986399,0.000433,0.986403,0.000662
8,random_forest,baseline,iot23,235490,5,1,{},26.330503,0.438842,15.948257,...,0.999788,0.000056,0.999098,0.000111,0.997739,0.000280,0.998556,0.000178,0.996709,0.000485
9,random_forest,baseline,cicids2017_unsw_nb15,470980,5,1,{},141.942059,2.365701,79.154763,...,0.990883,0.000327,0.993942,0.000202,0.985040,0.000493,0.990337,0.000321,0.989518,0.000406
